# 🏠 3. From Data to Decision
## Can Young People Still Afford to Live in Brighton?

Welcome to the main hands-on project notebook.

In this notebook, you are a **Junior Data Analyst** investigating a real local question:

> **Can young people still afford to live in Brighton?**

By the end, you will have DBT models, DBT tests, charts, a simple decision checker, an optional Streamlit app, and a short stakeholder summary.

# 🔗 Public Data Sources Used

This notebook uses a prepared teaching dataset based on public source figures and locally relevant assumptions.

Source links:

- ONS Brighton & Hove housing page: https://www.ons.gov.uk/visualisations/housingpriceslocal/E06000043/
- ONS Private rent and house prices bulletin: https://www.ons.gov.uk/economy/inflationandpriceindices/bulletins/privaterentandhousepricesuk/march2026
- GOV.UK National Minimum Wage rates: https://www.gov.uk/national-minimum-wage-rates
- Brighton & Hove Buses fare updates: https://www.buses.co.uk/fares-2026
- Brighton & Hove Buses tickets page: https://www.buses.co.uk/tickets

Source benchmarks:

- ONS reports average monthly private rent in Brighton & Hove was **£1,826 in March 2026**.
- ONS reports average one-bedroom rent in Brighton & Hove was **£1,198 in March 2026**.
- GOV.UK lists April 2026 minimum wage rates as **£8.00 apprentice**, **£10.85 for ages 18–20**, and **£12.71 for ages 21+**.
- Brighton & Hove Buses lists 2026 fares including **£3 single tickets** and citySAVER day-ticket options.

For reliability, we do not live-download data during the workshop. We create small CSV files from source-backed assumptions.

# ✅ Before You Start

Complete Notebook 2 first:

`02_environment/From_Zero_to_Working_DBT.ipynb`

That should have created `.venv/`, `dbt_project/`, `workshop.duckdb`, and a working DBT setup.

# 1️⃣ Setup: Find the Repo and Tools

This cell finds the workshop repo, DBT project, virtual environment, and database path.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import textwrap

OS_NAME = platform.system()

def find_repo_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    
    for candidate in candidates:
        if (
            (candidate / "README.md").exists()
            and (candidate / "requirements.txt").exists()
            and (candidate / "02_environment").exists()
        ):
            return candidate
            
  raise FileNotFoundError(
      "Could not find the workshop repo root. "
      "Open this notebook from inside the cloned dbt-workshop repo."
    )

repo_root = find_repo_root(Path.cwd())

dbt_project_dir = repo_root / "dbt_project"
venv_dir = repo_root / ".venv"

venv_python = venv_dir / (
    "Scripts/python.exe" if OS_NAME == "Windows" else "bin/python"
)

venv_pip = venv_dir / (
  "Scripts/pip.exe" if OS_NAME == "Windows" else "bin/pip"
)

venv_dbt = venv_dir / (
  "Scripts/dbt.exe" if OS_NAME == "Windows" else "bin/dbt"
)

profiles_dir = Path.home() / ".dbt"
database_path = (repo_root / "workshop.duckdb").resolve()

project_data_dir = repo_root / "03_project" / "data"
output_dir = repo_root / "03_project" / "output"
project_data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("Operating system:", OS_NAME)
print("Repo root:", repo_root)
print("DBT project exists:", dbt_project_dir.exists())
print("Virtual environment Python exists:", venv_python.exists())
print("Database:", database_path)
print("Project data folder:", project_data_dir)
print("Output folder:", output_dir)

## ✅ Checkpoint

You should see paths for the repo, DBT project, database, project data folder, and output folder.

If `dbt_project/` does not exist, complete Notebook 2 first.

# 2️⃣ Helper Function

This helper lets us run DBT commands from inside the notebook.

In [ ]:
def run_command(cmd, cwd=None, check=True):
    print("Running:", " ".join(str(x) for x in cmd))
    completed = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True
    )

    if completed.stdout.strip():
        print(completed.stdout)

    if completed.stderr.strip():
        print(completed.stderr)

    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")

    return completed

## ✅ Checkpoint

The helper function is ready.

# 3️⃣ Create the Project Datasets

We will create four CSV files:

- `affordability_rents.csv`
- `affordability_personas.csv`
- `affordability_living_costs.csv`
- `affordability_transport.csv`

In [ ]:
import pandas as pd

rents = pd.DataFrame([
    {"area": "Brighton Centre", "rent_option": "shared_room", "monthly_rent": 775, "notes": "Workshop estimate for shared room"},
    {"area": "Brighton Centre", "rent_option": "one_bed", "monthly_rent": 1250, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "North Laine", "rent_option": "shared_room", "monthly_rent": 800, "notes": "Workshop estimate for shared room"},
    {"area": "North Laine", "rent_option": "one_bed", "monthly_rent": 1280, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Hove", "rent_option": "shared_room", "monthly_rent": 850, "notes": "Workshop estimate for shared room"},
    {"area": "Hove", "rent_option": "one_bed", "monthly_rent": 1300, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Kemptown", "rent_option": "shared_room", "monthly_rent": 760, "notes": "Workshop estimate for shared room"},
    {"area": "Kemptown", "rent_option": "one_bed", "monthly_rent": 1160, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Brighton Marina", "rent_option": "shared_room", "monthly_rent": 825, "notes": "Workshop estimate for shared room"},
    {"area": "Brighton Marina", "rent_option": "one_bed", "monthly_rent": 1200, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
    {"area": "Moulsecoomb", "rent_option": "shared_room", "monthly_rent": 680, "notes": "Workshop estimate for shared room"},
    {"area": "Moulsecoomb", "rent_option": "one_bed", "monthly_rent": 1000, "notes": "Area estimate anchored to ONS one-bedroom benchmark"},
])

personas = pd.DataFrame([
    {"persona": "18-year-old apprentice", "age_group": "18-20", "employment_type": "Apprentice", "hourly_rate": 8.00, "hours_per_week": 37.5, "monthly_take_home_estimate": 1260, "notes": "Based on 2026 apprentice minimum wage; simplified monthly estimate"},
    {"persona": "19-year-old retail worker", "age_group": "18-20", "employment_type": "Retail / Hospitality", "hourly_rate": 10.85, "hours_per_week": 37.5, "monthly_take_home_estimate": 1660, "notes": "Based on 2026 18-20 minimum wage; simplified monthly estimate"},
    {"persona": "21-year-old full-time worker", "age_group": "21+", "employment_type": "Entry-level full-time", "hourly_rate": 12.71, "hours_per_week": 37.5, "monthly_take_home_estimate": 1900, "notes": "Based on 2026 National Living Wage; simplified monthly estimate"},
    {"persona": "23-year-old graduate analyst", "age_group": "21+", "employment_type": "Graduate role", "hourly_rate": 15.00, "hours_per_week": 37.5, "monthly_take_home_estimate": 2250, "notes": "Workshop scenario for early-career graduate income"},
])

living_costs = pd.DataFrame([
    {"cost_item": "food", "monthly_cost": 250, "category": "essential"},
    {"cost_item": "utilities_share", "monthly_cost": 120, "category": "essential"},
    {"cost_item": "mobile_phone", "monthly_cost": 25, "category": "essential"},
    {"cost_item": "internet_share", "monthly_cost": 25, "category": "essential"},
    {"cost_item": "basic_personal_spending", "monthly_cost": 150, "category": "essential"},
])

transport = pd.DataFrame([
    {"transport_option": "walk_or_cycle", "monthly_transport_cost": 20, "notes": "Low-cost active travel assumption"},
    {"transport_option": "bus_commuter", "monthly_transport_cost": 120, "notes": "Approx. two capped single fares per weekday"},
    {"transport_option": "daily_city_bus", "monthly_transport_cost": 136, "notes": "Approx. 20 adult citySAVER day tickets"},
])

rents.to_csv(project_data_dir / "affordability_rents.csv", index=False)
personas.to_csv(project_data_dir / "affordability_personas.csv", index=False)
living_costs.to_csv(project_data_dir / "affordability_living_costs.csv", index=False)
transport.to_csv(project_data_dir / "affordability_transport.csv", index=False)

print("✅ Created project CSV files:")
for path in sorted(project_data_dir.glob("affordability_*.csv")):
    print("-", path.name)

## ✅ Checkpoint

You should see four CSV files created.

# 4️⃣ Quick Look at the Data

Before transforming data, analysts inspect it.

In [ ]:
display(rents.head())
display(personas.head())
display(living_costs.head())
display(transport.head())

## ✅ Checkpoint

You should see four small tables.

# 5️⃣ Copy the CSV Files into DBT Seeds

DBT loads CSV files from the `seeds/` folder.

In [ ]:
seed_dir = dbt_project_dir / "seeds"
seed_dir.mkdir(parents=True, exist_ok=True)

for source_file in project_data_dir.glob("affordability_*.csv"):
    target_file = seed_dir / source_file.name
    if target_file.exists():
        print("✅ ", source_file.name, "file already exists")
        
    else:
        target_file.write_text(source_file.read_text(encoding="utf-8"), encoding="utf-8")
        print("Copied:", source_file.name, "→", target_file)
    
print("✅ CSV files copied into DBT seeds folder")

## ✅ Checkpoint

You should see each CSV copied into `dbt_project/seeds/`.

# 6️⃣ Create DBT Models

We will create a staging model and a decision-ready mart model.

In [ ]:
models_dir = dbt_project_dir / "models"
staging_dir = models_dir / "staging"
marts_dir = models_dir / "marts"
staging_dir.mkdir(parents=True, exist_ok=True)
marts_dir.mkdir(parents=True, exist_ok=True)

staging_model = staging_dir / "stg_affordability_inputs.sql"

if staging_model.exists():
    print("✅ stg_affordability_inputs.sql model file already exists")
    
else:
    staging_model.write_text(textwrap.dedent('''\
    with base_costs as (
        select sum(monthly_cost) as monthly_base_living_cost
        from {{ ref('affordability_living_costs') }}
    ),
    scenarios as (
        select
            p.persona,
            p.age_group,
            p.employment_type,
            p.monthly_take_home_estimate,
            r.area,
            r.rent_option,
            r.monthly_rent,
            t.transport_option,
            t.monthly_transport_cost,
            b.monthly_base_living_cost
        from {{ ref('affordability_personas') }} as p
        cross join {{ ref('affordability_rents') }} as r
        cross join {{ ref('affordability_transport') }} as t
        cross join base_costs as b
    )
    select * from scenarios
    '''), encoding="utf-8")


mart_model = marts_dir / "mart_affordability_scenarios.sql"

if mart_model.exists():
    print("✅ mart_affordability_scenarios.sql model file already exists")
    
else:
    mart_model.write_text(textwrap.dedent('''\
    with scenarios as (
        select * from {{ ref('stg_affordability_inputs') }}
    ),
    calculated as (
        select
            persona,
            age_group,
            employment_type,
            monthly_take_home_estimate,
            area,
            rent_option,
            monthly_rent,
            transport_option,
            monthly_transport_cost,
            monthly_base_living_cost,
            monthly_take_home_estimate - monthly_rent - monthly_transport_cost - monthly_base_living_cost as monthly_leftover,
            round((monthly_rent * 100.0) / monthly_take_home_estimate, 1) as rent_burden_pct
        from scenarios
    )
    select
        *,
        case
            when monthly_leftover >= 300 and rent_burden_pct <= 35 then 'Affordable'
            when monthly_leftover >= 0 then 'Tight'
            else 'Not affordable'
        end as affordability_status
    from calculated
    '''), encoding="utf-8")

print("✅ Created DBT models:")
print()
print("-", staging_model)
print()
print("-", mart_model)

## ✅ Checkpoint

You should now have staging and marts model files.

# 7️⃣ Add Data Quality Tests

Before using data for decisions, we add checks.

In [ ]:
macros_dir = dbt_project_dir / "macros"
macros_dir.mkdir(parents=True, exist_ok=True)

macro_file_pv = macros_dir / "test_positive_value.sql"

if macro_file_pv.exists():
    print("✅ Generic test positive value macro file already exists")
    
else:
    macro_file_pv.write_text(
        textwrap.dedent("""\
        {% test positive_value(model, column_name) %}
        
        select *
        from {{ model }}
        where {{ column_name }} <= 0
        
        {% endtest %}
        """), 
        encoding="utf-8"
    )

    print("✅ Generic test positive value macro created\n")


macro_file_nnv = macros_dir / "test_non_negative_value.sql"

if macro_file_nnv.exists():
    print("✅ Generic test non-negative value macro file already exists")
    
else:
    macro_file_nnv.write_text(
        textwrap.dedent("""\
        {% test non_negative_value(model, column_name) %}
        
        select *
        from {{ model }}
        where {{ column_name }} < 0
        
        {% endtest %}
        """), 
        encoding="utf-8"
    )

    print("✅ Generic test non-negative value macro created\n")

print("✅ Created custom DBT test macros")

## ✅ Checkpoint

You should see custom test macros created.

# 8️⃣ Create `schema.yml`

This file tells DBT which tests to run.

In [ ]:
schema_file = models_dir / "schema.yml"

schema_file.write_text(textwrap.dedent('''\
version: 2

seeds:
  - name: affordability_rents
    columns:
      - name: area
        tests:
          - not_null
      - name: rent_option
        tests:
          - not_null
          - accepted_values:
              values: ['shared_room', 'one_bed']
      - name: monthly_rent
        tests:
          - not_null
          - positive_value

  - name: affordability_personas
    columns:
      - name: persona
        tests:
          - not_null
          - unique
      - name: age_group
        tests:
          - not_null
          - accepted_values:
              values: ['18-20', '21+']
      - name: monthly_take_home_estimate
        tests:
          - not_null
          - positive_value

  - name: affordability_living_costs
    columns:
      - name: cost_item
        tests:
          - not_null
          - unique
      - name: monthly_cost
        tests:
          - not_null
          - non_negative_value

  - name: affordability_transport
    columns:
      - name: transport_option
        tests:
          - not_null
          - unique
      - name: monthly_transport_cost
        tests:
          - not_null
          - non_negative_value

models:
  - name: stg_affordability_inputs
    description: "Combined input scenarios for Brighton affordability analysis."
  - name: mart_affordability_scenarios
    description: "Decision-ready affordability scenarios."
    columns:
      - name: persona
        tests:
          - not_null
      - name: area
        tests:
          - not_null
      - name: monthly_leftover
        tests:
          - not_null
      - name: rent_burden_pct
        tests:
          - not_null
      - name: affordability_status
        tests:
          - not_null
          - accepted_values:
              values: ['Affordable', 'Tight', 'Not affordable']
'''), encoding="utf-8")

print("✅ Created schema.yml")
print()
print(schema_file.read_text(encoding="utf-8"))

## ✅ Checkpoint

The YAML should begin with `version: 2`, then `seeds:` at the far left.

# 9️⃣ Run DBT Debug

This checks the project, profile, and database connection.

In [ ]:
debug_result = run_command([venv_dbt, "debug", "--project-dir", dbt_project_dir, "--profiles-dir", profiles_dir], cwd=repo_root)
if debug_result.returncode == 0:
    print("✅ DBT debug passed")
else:
    raise RuntimeError("❌ DBT debug failed. Scroll up and read the output carefully.")

## ✅ Checkpoint

DBT should report that checks passed.

# 🔟 Load the Seeds into DuckDB

This loads the CSV files into the database.

In [ ]:
seed_result = run_command([venv_dbt, "seed", "--project-dir", dbt_project_dir, "--profiles-dir", profiles_dir], cwd=repo_root)
if seed_result.returncode == 0:
    print("✅ Seeds loaded successfully")
else:
    raise RuntimeError("❌ DBT seed loading failed. Scroll up and read the output carefully.")

## ✅ Checkpoint

You should see DBT load four seed files.

# 1️⃣1️⃣ Run the DBT Models

This builds the staging and mart tables.

In [ ]:
run_result = run_command([venv_dbt, "run", "--project-dir", dbt_project_dir, "--profiles-dir", profiles_dir], cwd=repo_root)
if run_result.returncode == 0:
    print("✅ DBT models built successfully")
else:
    raise RuntimeError("❌ DBT run failed. Scroll up and read the output carefully.")

## ✅ Checkpoint

You should see DBT build the staging and mart models.

# 1️⃣2️⃣ Run DBT Tests

Now DBT checks the rules in `schema.yml`.

In [ ]:
test_result = run_command([venv_dbt, "test", "--project-dir", dbt_project_dir, "--profiles-dir", profiles_dir], cwd=repo_root)
if test_result.returncode == 0:
    print("✅ DBT tests passed")
else:
    raise RuntimeError("❌ DBT tests failed. Scroll up and inspect which rule failed.")

## ✅ Checkpoint

Passing tests mean your data follows the rules we set.

# 1️⃣3️⃣ Load the Final Table into Python

Now we read the final DBT table from DuckDB.

In [ ]:
query_code = f'''
import duckdb
con = duckdb.connect(r"{database_path}")
df = con.execute("select * from mart_affordability_scenarios").fetchdf()
print(df.head(10).to_string(index=False))
print()
print("Rows:", len(df))
'''
query_result = run_command([venv_python, "-c", query_code], cwd=repo_root)

## ✅ Checkpoint

You should see rows from `mart_affordability_scenarios`.

# 1️⃣4️⃣ Analyse the Results

Now we use Python to explore the decision-ready table.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect(str(database_path))
affordability = con.execute("select * from mart_affordability_scenarios").fetchdf()
affordability.head()

## ✅ Checkpoint

You should see a dataframe.

# 1️⃣5️⃣ Summary: How Many Scenarios Are Affordable?

In [ ]:
status_summary = (
    affordability
    .groupby("affordability_status")
    .size()
    .reset_index(name="scenario_count")
    .sort_values("scenario_count", ascending=False)
)
status_summary

## ✅ Checkpoint

You should see counts for each affordability status.

# 1️⃣6️⃣ Chart: Affordability Status Counts

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(status_summary["affordability_status"], status_summary["scenario_count"])
plt.title("How many Brighton living scenarios are affordable?")
plt.xlabel("Affordability status")
plt.ylabel("Number of scenarios")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## ✅ Checkpoint

You should see a bar chart.

# 1️⃣7️⃣ Focus on Shared Rooms

For many young people, shared housing is the first realistic option.

In [ ]:
shared = affordability[affordability["rent_option"] == "shared_room"].copy()
shared_summary = (
    shared
    .groupby(["persona", "affordability_status"])
    .size()
    .reset_index(name="scenario_count")
)
shared_summary

## ✅ Checkpoint

You should see shared-room affordability counts by persona.

# 1️⃣8️⃣ Chart: Average Leftover by Persona

In [ ]:
leftover_by_persona = (
    affordability
    .groupby("persona")["monthly_leftover"]
    .mean()
    .sort_values()
    .reset_index()
)
plt.figure(figsize=(10, 5))
plt.barh(leftover_by_persona["persona"], leftover_by_persona["monthly_leftover"])
plt.title("Average monthly leftover by persona")
plt.xlabel("Average monthly leftover (£)")
plt.ylabel("Persona")
plt.tight_layout()
plt.show()
leftover_by_persona

## ✅ Checkpoint

Ask: which young person is most financially squeezed?

# 1️⃣9️⃣ Chart: Rent Burden by Rent Option

Rent burden means `rent ÷ income × 100`.

In [ ]:
rent_burden = (
    affordability
    .groupby(["persona", "rent_option"])["rent_burden_pct"]
    .mean()
    .reset_index()
)
pivot_burden = rent_burden.pivot(index="persona", columns="rent_option", values="rent_burden_pct")
pivot_burden.plot(kind="bar", figsize=(10, 5))
plt.title("Average rent burden by persona and rent option")
plt.xlabel("Persona")
plt.ylabel("Rent burden (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
pivot_burden

## ✅ Checkpoint

You should see that one-bed renting is much harder than shared renting.

# 2️⃣0️⃣ Decision Table: Best Options for Each Persona

In [ ]:
best_options = (
    affordability
    .sort_values(["persona", "monthly_leftover"], ascending=[True, False])
    .groupby("persona")
    .head(5)
    [["persona", "area", "rent_option", "transport_option", "monthly_take_home_estimate", "monthly_rent", "monthly_transport_cost", "monthly_leftover", "rent_burden_pct", "affordability_status"]]
)
best_options

## ✅ Checkpoint

You should see the top five options for each persona.

# 2️⃣1️⃣ Simple Dashboard: Could I Afford Brighton?

Change the values below and run the cell.

In [ ]:
selected_persona = "19-year-old retail worker"
selected_area = "Moulsecoomb"
selected_rent_option = "shared_room"
selected_transport_option = "bus_commuter"

choice = affordability[
    (affordability["persona"] == selected_persona)
    & (affordability["area"] == selected_area)
    & (affordability["rent_option"] == selected_rent_option)
    & (affordability["transport_option"] == selected_transport_option)
]

if choice.empty:
    print("No matching scenario found. Check your spelling.")
else:
    row = choice.iloc[0]
    print("🏠 Brighton affordability checker")
    print("--------------------------------")
    print("Persona:", row["persona"])
    print("Area:", row["area"])
    print("Rent option:", row["rent_option"])
    print("Transport:", row["transport_option"])
    print()
    print("Monthly take-home estimate: £", round(row["monthly_take_home_estimate"], 2))
    print("Monthly rent: £", round(row["monthly_rent"], 2))
    print("Monthly transport: £", round(row["monthly_transport_cost"], 2))
    print("Monthly base living costs: £", round(row["monthly_base_living_cost"], 2))
    print("Monthly leftover: £", round(row["monthly_leftover"], 2))
    print("Rent burden:", round(row["rent_burden_pct"], 1), "%")
    print()
    print("Status:", row["affordability_status"])

## ✅ Checkpoint

Try changing the selected values to test different scenarios.

# 2️⃣2️⃣ Optional: Create a Streamlit App

This creates a small app file that can be run later with:

```bash
streamlit run 03_project/brighton_affordability_app.py
```

In [ ]:
app_file = repo_root / "03_project" / "brighton_affordability_app.py"
app_template = '''
import duckdb
import streamlit as st

st.set_page_config(page_title="Brighton Affordability Checker", layout="centered")
st.title("🏠 Could You Afford Brighton?")
st.write("A simple workshop app using DBT output data.")

db_path = r"__DB_PATH__"
con = duckdb.connect(db_path)
df = con.execute("select * from mart_affordability_scenarios").fetchdf()

persona = st.selectbox("Choose a persona", sorted(df["persona"].unique()))
area = st.selectbox("Choose an area", sorted(df["area"].unique()))
rent_option = st.selectbox("Choose rent option", sorted(df["rent_option"].unique()))
transport_option = st.selectbox("Choose transport option", sorted(df["transport_option"].unique()))

selected = df[
    (df["persona"] == persona)
    & (df["area"] == area)
    & (df["rent_option"] == rent_option)
    & (df["transport_option"] == transport_option)
]

if not selected.empty:
    row = selected.iloc[0]
    st.metric("Monthly leftover", f"£{row['monthly_leftover']:.0f}")
    st.metric("Rent burden", f"{row['rent_burden_pct']:.1f}%")
    st.subheader(f"Status: {row['affordability_status']}")
    st.write("### Scenario details")
    st.dataframe(selected)

st.write("---")
st.write("This is a teaching app. Figures are simplified workshop estimates based on public-source benchmarks.")
'''
app_file.write_text(app_template.replace("__DB_PATH__", str(database_path)), encoding="utf-8")
print("✅ Streamlit app created:")
print(app_file)

## ✅ Checkpoint

You should see `03_project/brighton_affordability_app.py` created.

# 2️⃣3️⃣ Write Your Analyst Findings

Now move from code to communication.

In [ ]:
not_affordable_count = int((affordability["affordability_status"] == "Not affordable").sum())
total_count = len(affordability)
not_affordable_pct = round(not_affordable_count * 100 / total_count, 1)
most_stretched = leftover_by_persona.iloc[0]["persona"]
lowest_rent_area = rents.groupby("area")["monthly_rent"].mean().sort_values().index[0]

print("Automatic findings:")
print(f"1. {not_affordable_pct}% of scenarios are classified as not affordable.")
print(f"2. The most financially stretched persona is: {most_stretched}.")
print(f"3. The area with the lowest average rent in this teaching dataset is: {lowest_rent_area}.")

## ✅ Checkpoint

You should see three automatically generated findings.

# 2️⃣4️⃣ Frame Alternatives for a Stakeholder

A stakeholder does not just need facts. They need choices.

| Option | What it means | Who it helps |
|---|---|---|
| Shared housing support | Help young people find safe shared housing | Apprentices and lower-paid workers |
| Transport support | Reduce commuting cost pressure | People living farther from central Brighton |
| Local wage partnerships | Encourage employers to pay above minimum wage | Young full-time workers |
| Affordable starter housing | Increase access to lower-cost rentals | Young people trying to live independently |
| Career pathway support | Help young residents move into higher-paid digital roles | Students and early-career workers |

# 2️⃣5️⃣ Create a Stakeholder Summary

This cell writes a short markdown report you can keep.

In [ ]:
summary_file = output_dir / "brighton_affordability_summary.md"
summary_text = f"""# Brighton Affordability Analysis Summary

## Question

Can young people still afford to live in Brighton?

## Data Used

This analysis used prepared workshop datasets based on public-source benchmarks for:

- Brighton rents
- young worker income scenarios
- essential living costs
- transport cost scenarios

## Key Findings

1. {not_affordable_pct}% of modelled scenarios were classified as **not affordable**.
2. The most financially stretched persona was: **{most_stretched}**.
3. The area with the lowest average rent in this teaching dataset was: **{lowest_rent_area}**.

## Decision Alternatives

Stakeholders could consider:

1. Support safe shared housing options for young people.
2. Improve transport affordability for young workers.
3. Work with local employers on better early-career pay.
4. Expand affordable starter housing options.
5. Help young people access higher-paid digital and analytical career pathways.

## Important Caveat

This is a workshop model. The data is simplified for teaching and should be replaced with a fuller official dataset before making real policy decisions.
"""
summary_file.write_text(summary_text, encoding="utf-8")
print("✅ Summary report written:")
print(summary_file)
print()
print(summary_text)

## ✅ Checkpoint

You should now have:

`03_project/output/brighton_affordability_summary.md`

# 🎉 Final Reflection

You have completed a full data analysis process:

```text
Question → Data → DBT Transformation → Tests → Analysis → Visuals → Decision Alternatives
```

## CV-style description

> Built a Brighton housing affordability analysis using Python, DuckDB and DBT. Modelled young-person income scenarios, rent options and living costs to identify affordability pressures and decision alternatives for local stakeholders.

## Future extensions

- use live official datasets
- include more Brighton neighbourhoods
- add council tax estimates
- add salary data by occupation
- compare Brighton with nearby towns
- host the Streamlit app as a portfolio project